In [ ]:
import matplotlib.pylab as plt
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

In [ ]:
from gen import (
    create_heating_on, 
    create_cooling_on, 
    create_occ_profile, 
    generate_heating_energy,
    generate_cooling_energy,
    total_interlock
)
from metrics import cvrmse, nmbe

In [ ]:
BOOST_PARAMS = {
    "iterations": 200,
    "depth": 3,
    "bootstrap_type": "Bayesian",
    "task_type": "CPU",
    "has_time": True,
    "allow_writing_files": False,
    "verbose": False,
}

### Before Retrofit (2023)

In [ ]:
weather_2023 = pd.read_csv("../data/inputs/weather-2023.csv")
weather_2023["time"] = pd.to_datetime(weather_2023["time"])
weather_2023 = weather_2023.set_index("time")
weather_2023.head(2)

In [ ]:
heating_on_2023 = create_heating_on(weather_2023["temperature"])
cooling_on_2023 = create_cooling_on(weather_2023["temperature"])

In [ ]:
occupancy_2023 = create_occ_profile(weather_2023.index)

In [ ]:
monthly_factors = {
    1: 0.9,
    2: 0.95,
    3: 1.0,
    4: 1.05,
    5: 1.1,
    6: 1.15,
    7: 0.75,
    8: 0.6,
    9: 1.1,
    10: 1.05,
    11: 1.0,
    12: 0.85,
}

In [ ]:
heating_energy_2023, occupants_2023 = generate_heating_energy(
    weather_2023["temperature"],
    heating_on_2023,
    occupancy_2023,
    UA=3,
    zero_day_probability=0,
    monthly_factors=monthly_factors
)

In [ ]:
fig = plt.figure(figsize=(12, 3))
layout = (1, 1)
ax = plt.subplot2grid(layout, (0, 0))

occupants_2023.groupby(lambda x: x.date()).mean().plot(ax=ax)

In [ ]:
cooling_energy_2023, _ = generate_cooling_energy(
    weather_2023["temperature"],
    cooling_on_2023,
    occupancy_2023,
    UA=3.5,
    zero_day_probability=0,
    monthly_factors=monthly_factors
)

In [ ]:
results_2023 = total_interlock(
    heating_energy_2023,
    cooling_energy_2023,
    heating_on_2023,
    cooling_on_2023,
    occupancy_2023,
    weather_2023["temperature"],
)

In [ ]:
fig = plt.figure(figsize=(12, 3))
layout = (1, 1)
ax = plt.subplot2grid(layout, (0, 0))

ax.scatter(weather_2023, results_2023["total"], s=2, alpha=0.4)
ax.set_xlabel("Temperature [°C]")
ax.set_ylabel("Energy [kWh]")
ax.set_title("Before upgrade");

In [ ]:
temperature = weather_2023[["temperature"]]
consumption = results_2023["total"]

#### Full year of data

In [ ]:
calendar_features = pd.DataFrame(
    {
        "hour_of_day": temperature.index.hour.values,
        "day_of_week": temperature.index.dayofweek.values,
        "week_of_year": temperature.index.isocalendar().week.values
    },
    index=temperature.index
)

In [ ]:
temperature_features = pd.DataFrame(
    {
        "temperature": temperature["temperature"],
        "temp_L_1D": temperature["temperature"].rolling("24h", min_periods=1).mean().reindex(temperature.index).bfill().ffill(),
        "temp_L_2D": temperature["temperature"].shift(freq="24h").rolling("24h", min_periods=1).mean().reindex(temperature.index).bfill().ffill()
    },
)

In [ ]:
features = pd.concat([calendar_features, temperature_features], axis=1)

In [ ]:
model = CatBoostRegressor(**BOOST_PARAMS)

In [ ]:
model = model.fit(features, consumption)

In [ ]:
pred = pd.Series(model.predict(features), index=features.index)

In [ ]:
print("CV(RMSE):", cvrmse(consumption, pred))
print("NMBE:", nmbe(consumption, pred))

#### 4 months of data

In [ ]:
months = [1, 2, 8, 9] # January, February, August, September

In [ ]:
temperature = temperature[np.isin(temperature.index.month, months)]

In [ ]:
consumption = consumption.loc[temperature.index]

In [ ]:
consumption.to_frame("consumption").to_csv("../data/generated/pre-retrofit/consumption-2023.csv")

In [ ]:
print("Hours seen:", np.unique(consumption.index.hour))
print("Week days seen:", np.unique(consumption.index.dayofweek))
print("Months seen:", np.unique(consumption.index.month))

In [ ]:
calendar_features = pd.DataFrame(
    {
        "hour_of_day": temperature.index.hour.values,
        "day_of_week": temperature.index.dayofweek.values,
        "week_of_year": temperature.index.isocalendar().week.values
    },
    index=temperature.index
)

In [ ]:
temperature_padded = temperature["temperature"].reindex(weather_2023.index)

temperature_features = pd.DataFrame(
    {
        "temperature": temperature["temperature"],
        "temp_L_1D": temperature_padded.rolling("24h", min_periods=1).mean().reindex(temperature.index).bfill().ffill(),
        "temp_L_2D": temperature_padded.shift(freq="24h").rolling("24h", min_periods=1).mean().reindex(temperature.index).bfill().ffill()
    },
)

In [ ]:
features = pd.concat([calendar_features, temperature_features], axis=1)

In [ ]:
model = CatBoostRegressor(**BOOST_PARAMS)
model = model.fit(features, consumption)

### After Retrofit (2024)

In [ ]:
weather_2024 = pd.read_csv("../data/inputs/weather-2024.csv")
weather_2024["time"] = pd.to_datetime(weather_2024["time"])
weather_2024 = weather_2024.set_index("time")
weather_2024.head(2)

In [ ]:
heating_on_2024 = create_heating_on(weather_2024["temperature"])
cooling_on_2024 = create_cooling_on(weather_2024["temperature"])

In [ ]:
occupancy_2024 = create_occ_profile(weather_2024.index)

In [ ]:
heating_energy_2024, _ = generate_heating_energy(
    weather_2024["temperature"],
    heating_on_2024,
    occupancy_2024,
    UA=2,
    zero_day_probability=0,
    monthly_factors=monthly_factors
)

In [ ]:
cooling_energy_2024, _ = generate_cooling_energy(
    weather_2024["temperature"],
    cooling_on_2024,
    occupancy_2024,
    UA=3,
    zero_day_probability=0,
    monthly_factors=monthly_factors
)

In [ ]:
results_2024 = total_interlock(
    heating_energy_2024,
    cooling_energy_2024,
    heating_on_2024,
    cooling_on_2024,
    occupancy_2024,
    weather_2024["temperature"],
)

In [ ]:
fig = plt.figure(figsize=(12, 3))
layout = (1, 1)
ax = plt.subplot2grid(layout, (0, 0))

ax.scatter(weather_2024, results_2024["total"], s=2, alpha=0.4)
ax.set_xlabel("Temperature [°C]")
ax.set_ylabel("Energy [kWh]")
ax.set_title("After upgrade");

In [ ]:
results_2024[["total"]].rename(columns={"total": "consumption"}).to_csv("../data/generated/post-retrofit/consumption-2024.csv")

In [ ]:
temperature = weather_2024[["temperature"]]
consumption = results_2024["total"]

In [ ]:
calendar_features = pd.DataFrame(
    {
        "hour_of_day": temperature.index.hour.values,
        "day_of_week": temperature.index.dayofweek.values,
        "week_of_year": temperature.index.isocalendar().week.values
    },
    index=temperature.index
)

In [ ]:
temperature_features = pd.DataFrame(
    {
        "temperature": temperature["temperature"],
        "temp_L_1D": temperature["temperature"].rolling("24h", min_periods=1).mean().reindex(temperature.index).bfill().ffill(),
        "temp_L_2D": temperature["temperature"].shift(freq="24h").rolling("24h", min_periods=1).mean().reindex(temperature.index).bfill().ffill()
    },
)

In [ ]:
features = pd.concat([calendar_features, temperature_features], axis=1)

In [ ]:
pred = pd.Series(model.predict(features), index=features.index)

In [ ]:
savings = pred - consumption

In [ ]:
print("Total savings:", float(savings.sum()))

### True counterfactual

In [ ]:
heating_energy_cf, _ = generate_heating_energy(
    weather_2024["temperature"],
    heating_on_2024,
    occupancy_2024,
    UA=3,
    zero_day_probability=0,
    noise_std=0,
    monthly_factors=monthly_factors
)

In [ ]:
cooling_energy_cf, _ = generate_cooling_energy(
    weather_2024["temperature"],
    cooling_on_2024,
    occupancy_2024,
    UA=3.5,
    zero_day_probability=0,
    noise_std=0,
    monthly_factors=monthly_factors
)

In [ ]:
results_cf = total_interlock(
    heating_energy_cf,
    cooling_energy_cf,
    heating_on_2024,
    cooling_on_2024,
    occupancy_2024,
    weather_2024["temperature"],
)

In [ ]:
results_cf[["total"]].rename(columns={"total": "consumption"}).to_csv("../data/generated/post-retrofit/counterfactual.csv")

In [ ]:
real_savings = results_cf["total"] - consumption

In [ ]:
fig = plt.figure(figsize=(12, 3))
layout = (1, 1)
ax = plt.subplot2grid(layout, (0, 0))

savings.cumsum().plot(ax=ax)
real_savings.cumsum().plot(ax=ax)
ax.legend(["Estimated savings", "Actual savings"])

In [ ]:
print("Total real savings:", float(real_savings.sum()))

In [ ]:
meval = pd.read_csv("../data/test-total-savings-kWh.csv")
meval["meval_timestamp"] = pd.to_datetime(meval["meval_timestamp"])
meval = meval.set_index("meval_timestamp")

In [ ]:
fig = plt.figure(figsize=(12, 3))
layout = (1, 1)
ax = plt.subplot2grid(layout, (0, 0))

savings.cumsum().plot(ax=ax)
meval["meanCumulativeSavings"].plot(ax=ax)
real_savings.cumsum().plot(ax=ax, color="black", style="--")
ax.legend(["TOWT-estimated savings", "Meval-estimated savings", "Actual savings"])